# 2026-09-01_5 AUCell violin 重排（排图用）
需求：每个 celltype × direction × subclass 面板各自独立 y 轴；
      相同 direction × subclass 的全部 celltype 合并输出到同一个 PDF（减少文件数）；
      每个 PDF 内：胚胎、成年各一页（分数跨期不可比，不合并为一页）。
直接复用 2026-09-01_2 打分产物（两期 meta+score RDS），不重新打分。

2026-09-04 追加：sharedY 版本（`_sharedY` 后缀）——同一 celltype 面板在胚胎页与成年页 y 轴范围一致，便于两期对比；原版（freeY）输出不变。


In [ ]:
suppressPackageStartupMessages({
  library(dplyr)
  library(tidyr)
  library(readr)
  library(ggplot2)
})

date_str <- "2026-09-01_5"
data_dir <- "/project/work/2026-09-01_DEG_summary/Data/"
plot_dir <- "/project/work/2026-09-01_DEG_summary/Plot/"

celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
subclass_order <- c(
  "Developmental maintained",
  "Developmental aging-lost",
  "Adult acquired and maintained"
)
subclass_short <- c(
  "Developmental maintained" = "DM",
  "Developmental aging-lost" = "DAL",
  "Adult acquired and maintained" = "AAM"
)
direction_order <- c("Macular direction", "Peripheral direction")
stage_order <- c("Early", "Mid", "Late", "Young", "Old")
tissue_order <- c("central", "peripheral")
cols.tissue.merged <- c(central = "#6A3D9A", peripheral = "#E69F00")

signature_key <- read_csv(
  file.path(data_dir, "2026-09-01_2_AUCell_signature_definition.csv"),
  show_col_types = FALSE
)


In [ ]:
# ---- 1. 两期 meta+score RDS 转长表（与 2026-09-01_2 相同的整理逻辑） --------
plot_long <- bind_rows(lapply(c("Embryonic", "Adult"), function(period) {
  meta_df <- readRDS(file.path(data_dir, paste0("2026-09-01_2_AUCell_", period, "_meta_data.rds")))
  tissue_raw <- if ("tissue" %in% colnames(meta_df)) meta_df$tissue else meta_df$tissue_group
  meta_df %>%
    mutate(
      period = period,
      donor_age = as.character(donor_age),
      celltype = as.character(celltype),
      tissue = case_when(
        tolower(tissue_raw) %in% c("fovea", "macula", "central") ~ "central",
        tolower(tissue_raw) == "peripheral" ~ "peripheral",
        TRUE ~ NA_character_
      ),
      stage = case_when(
        donor_age %in% c("70days", "79days", "87days", "91days") ~ "Early",
        donor_age %in% c("100days", "103days", "116days") ~ "Mid",
        donor_age %in% c("137days", "141days", "142days", "162days", "165days") ~ "Late",
        donor_age %in% c("21years", "25years", "30years", "32years") ~ "Young",
        donor_age %in% c("67years", "70years", "75years", "86years") ~ "Old"
      )
    ) %>%
    filter(celltype %in% celltype_order, !is.na(tissue), !is.na(stage)) %>%
    select(period, celltype, donor_age, stage, tissue, starts_with("AUCell_")) %>%
    pivot_longer(starts_with("AUCell_"), names_to = "signature_id", values_to = "AUCell_score") %>%
    mutate(signature_id = sub("^AUCell_", "", signature_id)) %>%
    filter(!is.na(AUCell_score))
})) %>%
  left_join(
    signature_key %>% select(signature_id, signature_celltype = celltype, subclass, direction_class),
    by = "signature_id"
  ) %>%
  filter(celltype == signature_celltype) %>%
  mutate(
    celltype = factor(celltype, levels = celltype_order),
    tissue = factor(tissue, levels = tissue_order),
    stage_tissue = factor(paste(stage, tissue, sep = "\n"),
                          levels = paste(rep(stage_order, each = 2), tissue_order, sep = "\n"))
  )
cat("长表记录数：", nrow(plot_long), "\n")


## 3. 每个 subclass × direction × celltype 单独一张 PDF
- 胚胎与成年横向并排分面（`facet_grid(~period, scales="free_x", space="free_x")`），每期只显示本期 stage
- 两期共享同一 y 轴（`facet_grid` 默认固定 y 轴）
- 风格参考 p_diverging：`theme_bw`、灰色 strip、无竖网格、x 轴 45° 旋转
- 宽度按各期实际 x 槽数自适应（0.44 inch/槽 + 2.8 inch 边距），高度固定 3.8 inch
- 命名：`{date_str}_AUCell_violin_byCT_{subclass}_{direction}_{celltype}.pdf`

In [ ]:
# ---- 3. 每个 subclass × direction × celltype 单独一张 PDF（双期横排分面，共享 Y）--------
# 风格参考 p_diverging：theme_bw / 灰色 strip / 无竖网格 / x 轴 45° 旋转
# 宽度按各期 x 槽数自适应（0.44 inch/槽 + 2.8 inch 边距），高度固定 3.8 inch

for (one_subclass in subclass_order) {
  for (one_direction in direction_order) {
    dir_tag   <- if (one_direction == "Macular direction") "Central" else "Peripheral"
    dir_label <- if (one_direction == "Macular direction") "Central direction" else "Peripheral direction"
    sub_df  <- plot_long %>%
      filter(subclass == one_subclass, direction_class == one_direction)

    for (one_ct in celltype_order) {
      ct_df <- filter(sub_df, celltype == one_ct)
      if (nrow(ct_df) == 0) next

      # 确保 period 顺序为 Embryonic（左）→ Adult（右）
      ct_df <- ct_df %>% mutate(period = factor(period, levels = c("Embryonic", "Adult")))

      # 自适应宽度：按各期实际出现的 stage_tissue 类别数估算
      n_x_emb   <- ct_df %>% filter(period == "Embryonic") %>%
                     pull(stage_tissue) %>% droplevels() %>% nlevels()
      n_x_adult <- ct_df %>% filter(period == "Adult") %>%
                     pull(stage_tissue) %>% droplevels() %>% nlevels()
      pdf_w <- (n_x_emb + n_x_adult) * 0.44 + 2.8

      pdf_file <- file.path(plot_dir, paste0(
        date_str, "_AUCell_violin_byCT_",
        subclass_short[[one_subclass]], "_", dir_tag, "_", one_ct, ".pdf"
      ))

      p <- ggplot(ct_df, aes(x = stage_tissue, y = AUCell_score, fill = tissue)) +
        geom_violin(scale = "width", linewidth = 0.2, trim = TRUE) +
        facet_grid(~ period, scales = "free_x", space = "free_x") +
        scale_fill_manual(values = cols.tissue.merged, drop = FALSE) +
        labs(
          title    = paste0(subclass_short[[one_subclass]], " - ",
                            dir_label, " - ", one_ct),
          x = NULL, y = "AUCell score", fill = "Tissue"
        ) +
        coord_cartesian(clip = "off") +
        theme_bw(base_size = 9) +
        theme(
          axis.text.x        = element_text(angle = 45, hjust = 1,
                                            size = 7.5, color = "black"),
          axis.text.y        = element_text(size = 7.5),
          panel.grid.major.x = element_blank(),
          panel.grid.minor   = element_blank(),
          strip.background   = element_rect(fill = "gray90", color = "gray70"),
          strip.text         = element_text(face = "bold", size = 9),
          legend.position    = "right",
          legend.key.size    = unit(0.4, "cm"),
          legend.text        = element_text(size = 7.5),
          legend.title       = element_text(size = 8),
          plot.title         = element_text(face = "bold", hjust = 0.5, size = 9),
          plot.margin        = margin(4, 4, 4, 4)
        )

      pdf(pdf_file, width = pdf_w, height = 2.8, useDingbats = FALSE)
      print(p)
      dev.off()
      cat("已输出：", basename(pdf_file), "\n")
    }
  }
}
cat("全部完成。\n")